In [ ]:
!pip -q install tensorflow-hub pillow matplotlib requests

In [ ]:
import tensorflow as tf
****************
import numpy as np
import matplotlib.pyplot as plt
****************
import requests
from PIL import Image
from io import BytesIO

print('TensorFlow version:', tf.__version__)

In [ ]:
# One COCO image
url = 'https://huggingface.co/datasets/hf-internal-testing/fixtures-coco/resolve/main/val2017/000000039769.jpg'
response = requests.get(url, timeout=60)
response.raise_for_status()
image = np.array(Image.open(BytesIO(response.content)).convert('RGB'))

plt.figure(figsize=(8, 5))
plt.imshow(image)
plt.axis('off')
plt.title('Original image')
plt.show()

In [ ]:
# Load the pretrained model from TensorFlow Hub
model_url = 'https://tfhub.dev/tensorflow/faster_rcnn/resnet50_v1_640x640/1'
****************

# Faster R-CNN expects a batch of uint8 images
predictions = model(tf.convert_to_tensor(image[None, ...], dtype=tf.uint8))
print('Outputs:', list(predictions.keys()))

In [ ]:
def show_boxes(image, predictions, threshold=0.5):
****************
    scores = predictions['detection_scores'][0].numpy()
    classes = predictions['detection_classes'][0].numpy().astype(int)
    height, width = image.shape[:2]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.imshow(image)

    for box, score, label in zip(boxes, scores, classes):
****************
        top, left, bottom, right = box
        x, y = left * width, top * height
        w = (right - left) * width
        h = (bottom - top) * height
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2))
        ax.text(x, y, f'Class {label}: {score:.2f}', color='black', bbox=dict(facecolor='lime', alpha=0.8))

    ax.axis('off')
    plt.show()

show_boxes(image, predictions, threshold=0.5)

In [ ]:
import requests, re

url = "https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/data/mscoco_label_map.pbtxt"
text = requests.get(url).text
coco_names = {int(i): name for i, name in re.findall(r'id:\s*(\d+)\s+display_name:\s*"([^"]+)"', text)}
****************


In [ ]:
def show_boxes(image, predictions, threshold=0.5):
    boxes = predictions['detection_boxes'][0].numpy()
    scores = predictions['detection_scores'][0].numpy()
    classes = predictions['detection_classes'][0].numpy().astype(int)
    height, width = image.shape[:2]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.imshow(image)

    for box, score, label in zip(boxes, scores, classes):
        if score < threshold: continue
        top, left, bottom, right = box
        x, y = left * width, top * height
        w, h = (right - left) * width, (bottom - top) * height
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2))
****************
        ax.text(x, y, f'{name}: {score:.2f}', color='black', bbox=dict(facecolor='lime', alpha=0.8))

    ax.axis('off')
    plt.show()

****************


In [ ]:
def show_boxes(image, predictions, threshold=0.5, only_class=None):
    boxes = predictions['detection_boxes'][0].numpy()
    scores = predictions['detection_scores'][0].numpy()
    classes = predictions['detection_classes'][0].numpy().astype(int)
    height, width = image.shape[:2]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.imshow(image)

    for box, score, label in zip(boxes, scores, classes):
        if score < threshold or (only_class is not None and label != only_class): continue
****************
        x, y = left * width, top * height
        w, h = (right - left) * width, (bottom - top) * height
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2))
        name = coco_names.get(label, f'Class {label}')
        ax.text(x, y, f'{name}: {score:.2f}', color='black', bbox=dict(facecolor='lime', alpha=0.8))

    ax.axis('off')
    plt.show()

****************


In [ ]:
****************


In [ ]:
import torch
****************
from PIL import Image

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
****************
torch_image = Image.fromarray(image)
torch_input = weights.transforms()(torch_image)

with torch.inference_mode():
    torch_predictions = torch_model([torch_input])[0]

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(image)

for box, score, label in zip(torch_predictions['boxes'], torch_predictions['scores'], torch_predictions['labels']):
    if score < 0.5: continue
    x1, y1, x2, y2 = box.tolist()
    name = weights.meta['categories'][label.item()]
    ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='cyan', linewidth=2))
    ax.text(x1, y1, f'{name}: {score:.2f}', color='black', bbox=dict(facecolor='cyan', alpha=0.8))

ax.set_title('PyTorch Faster R-CNN')
ax.axis('off')
plt.show()

In [ ]:
from pathlib import Path
from io import BytesIO
import zipfile

folder = Path('PennFudanPed')

if not folder.exists():
    url = 'https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip'
    with zipfile.ZipFile(BytesIO(requests.get(url).content)) as z: z.extractall('.')

for path in sorted((folder / 'PNGImages').glob('*.png'))[:3]:
    photo = np.array(Image.open(path).convert('RGB'))
    mask = np.array(Image.open(folder / 'PedMasks' / f'{path.stem}_mask.png'))
    actual = len(np.unique(mask)) - 1
****************
    scores = result['detection_scores'][0].numpy()
    classes = result['detection_classes'][0].numpy().astype(int)
    predicted = np.sum((classes == 1) & (scores >= 0.5))
    print(f'{path.name}: Actual = {actual}, Predicted = {predicted}')
    show_boxes(photo, result, threshold=0.5, only_class=1)

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

url = 'https://tfhub.dev/tensorflow/mask_rcnn/inception_resnet_v2_1024x1024/1'
mask_model = hub.load(url)


In [ ]:
****************
print(mask_predictions.keys())

In [ ]:
def show_masks(image, predictions, threshold=0.5, only_class=None):
    boxes = predictions['detection_boxes'][0].numpy()
    scores = predictions['detection_scores'][0].numpy()
    classes = predictions['detection_classes'][0].numpy().astype(int)
    masks = predictions['detection_masks'][0].numpy()
    height, width = image.shape[:2]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.imshow(image)
    colors = ['lime', 'cyan', 'yellow', 'magenta']
    count = 0

    for box, score, label, mask in zip(boxes, scores, classes, masks):
        if score < threshold or (only_class is not None and label != only_class): continue
        top, left, bottom, right = box
        x1, y1, x2, y2 = int(left * width), int(top * height), int(right * width), int(bottom * height)
****************
        if x2 <= x1 or y2 <= y1: continue
        resized_mask = tf.image.resize(mask[..., None], (y2-y1, x2-x1)).numpy()[:, :, 0] > 0.5
        full_mask = np.zeros((height, width), dtype=bool)
        full_mask[y1:y2, x1:x2] = resized_mask
        ax.imshow(np.ma.masked_where(~full_mask, full_mask), cmap='spring', alpha=0.45, vmin=0, vmax=1)
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=colors[count % 4], linewidth=2))
        name = coco_names.get(label, f'Class {label}')
        ax.text(x1, y1, f'{name}: {score:.2f}', color='black', bbox=dict(facecolor=colors[count % 4], alpha=0.8))
        count += 1

    ax.axis('off')
    plt.show()
    print(f'Objects detected: {count}')

show_masks(image, mask_predictions, threshold=0.5, only_class=None)

In [ ]:
****************
